In [2]:
# % pip install medmnist
# to download the dataset for the medmnist package

In [5]:
import torch 
import numpy as np
import matplotlib.pyplot as plt
import torch 
import torch.nn as nn 
import torchvision.models as models

In [4]:
from medmnist.dataset import PathMNIST
from medmnist import INFO


# Load the dataset
train_dataset = PathMNIST(split='train', download=True)
val_dataset = PathMNIST(split='val', download=True)
test_dataset = PathMNIST(split='test', download=True)

# Get the info of the dataset
info = INFO['pathmnist']

100%|██████████| 205615438/205615438 [00:23<00:00, 8818813.25it/s] 


Using downloaded and verified file: C:\Users\Windows\.medmnist\pathmnist.npz
Using downloaded and verified file: C:\Users\Windows\.medmnist\pathmnist.npz


### EDA

# Architecture Building 

In [ ]:
class pretrainVGG16_PathMNIST(nn.modules):
    def __init__(self, num_classes=9):
        # Override 
        super(pretrainVGG16_PathMNIST, self).__init__()
        
        # Load the pretrained VGG16
        self.vgg16 = models.vgg16(pretrained=True)

        # Freeze the layers of VGG16
        for para in self.vgg16.parameters():
            para.requires_grad = False

        # Replace the last fully connected layer with a new one
        # Make it fits the number of PM classes 
        in_features = self.vgg16.classifier[6].in_features
        self.vgg16.classifier[6] = nn.Linear(
            in_features=in_features,
            out_features=num_classes
        ) 

    def forward(self, x):
        output = self.vgg16(x)
        return output



In [ ]:
class VGG16_PathMNIST(nn.modules):
    def __init__(self, num_classes=9):
        # Override 
        super(VGG16_PathMNIST, self).__init__()

        # VGG16 convolution blocks
        self.features = nn.Sequential(

            # Block 1
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 2
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 3
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 4
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Block 5
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )

        # classifier
        self.classifier = nn.Sequential(
            nn.Linear(512*7*7, 4096),
            nn.ReLU(True),
            nn.Dropout(),

            nn.Linear(4096, 4096),
            nn.ReLU(True),
            nn.Dropout(),

            nn.Linear(4096, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        output = self.classifier(x)
        return output



# Training and Hyper-parameter tuning 

# Evaluation with Validation

# Performace Metrics